In [18]:
import joblib
from tensorflow.keras.models import load_model
import numpy as np

In [ ]:
tk=joblib.load("yelp-tokenizer-10k.pkl")

model=load_model("nwp_chk_10k_100d_384neuron_gru.keras")

I0000 00:00:1795328785.575231   11091 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2132 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [29]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 100, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 100, 384)       │       559,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 100, 256)       │       493,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 128)            │       148,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,522,994 (40.14 MB)

 Trainable params: 3,507,664 (13.38 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,015,330 (26.76 MB)

In [6]:
from datasets import load_dataset

ds = load_dataset("Yelp/yelp_review_full")
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [7]:
import pandas as pd
samp=pd.DataFrame(ds["test"])
samp=samp[["text"]]
samp[:2]

,text
0,I got 'new' tires from them and within two wee...
1,Don't waste your time. We had two different p...


In [66]:
sample=samp["text"].sample(1).values[0]

sample

"great salads and their fries and chips are to die for.My friend's love their grilled cheese (not on the menu) and i've always had awesome. service. Great place for lunch and a good beer!"

In [70]:
sample[:28]

'great salads and their fries'

In [24]:
context=tk.texts_to_sequences(["service was"])[0]
context

[49, 7]

In [32]:
x=np.zeros((1,100),dtype=np.int32)
x

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=int32)

In [34]:
x[0,:len(context)]=context
x

array([[49,  7,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0]], dtype=int32)

In [38]:
y_prob=model.predict(x,verbose=0)[0]
y_prob

array([3.7021053e-11, 1.1066033e-02, 2.7992257e-03, ..., 2.8739743e-14,
       3.9885582e-14, 5.2500706e-16], shape=(10000,), dtype=float32)

In [39]:
labs=np.argmax(y_prob,axis=0)
labs

np.int64(30)

In [42]:
tk.index_word[labs]

'good'

In [51]:
np.argsort(y_prob,axis=0,descending=True)[:5]

array([ 30,  47,  42, 159, 371])

In [ ]:
max_len=100
def next_word_predict_n(text, generate=20, no_repeat_n_gram_size=3,
                        top_k=20, repetition_window=15):

    text = text.lower()

    for _ in range(generate):
        tokens = tk.texts_to_sequences([text])[0]
        tokens = tokens[-max_len:]

        x = np.zeros((1, max_len), dtype=np.int32)
        x[0, :len(tokens)] = tokens

        y_prob = model.predict(x, verbose=0)[0].copy()

        # Penalize recently generated words
        recent_tokens = set(tokens[-repetition_window:])

        for idx in recent_tokens:
            if idx > 0:
                y_prob[idx] *= 0.5

        # Identify repeated n-grams
        blocked_tokens = set()

        if len(tokens) >= no_repeat_n_gram_size - 1:
            prefix = tuple(tokens[-(no_repeat_n_gram_size - 1):])

            for i in range(len(tokens) - no_repeat_n_gram_size + 1):
                ngram = tuple(tokens[i:i + no_repeat_n_gram_size])

                if ngram[:-1] == prefix:
                    blocked_tokens.add(ngram[-1])

        # Block repeated n-grams
        for idx in blocked_tokens:
            y_prob[idx] = 0

        # Select highest-scoring available candidate
        probable_indices = np.argsort(y_prob)[::-1]

        valid_indices=[int(idx) for idx in probable_indices if idx not in (0,1) and y_prob[idx]>0][:top_k]

        if not valid_indices:
            break

        predicted_index = valid_indices[0]
        predicted_word = tk.index_word.get(predicted_index)

        if predicted_word is None:
            break

        text += " " + predicted_word

    return text


In [77]:
print(sample)
next_word_predict_n("Great place for lunch")

great salads and their fries and chips are to die for.My friend's love their grilled cheese (not on the menu) and i've always had awesome. service. Great place for lunch and a good beer!


'great place for lunch and the food is good but the service is very good and the service was great and the prices are'

In [78]:
next_word_predict_n("i would definitely recommend")

'i would definitely recommend this place to the food and the service is good but the food is great but the service was great'

In [79]:
next_word_predict_n("the service was good but")

'the service was good but i would have been to the food and it was a little pricey for the price of the food but'

In [80]:
prompts = [
    "the atmosphere was",
    "the staff were very",
    "the prices were",
    "the location is",
    "the food was"
]

for prompt in prompts:
    print("\nInput:", prompt)
    print("Output:", next_word_predict_n(prompt, generate=12))


Input: the atmosphere was
Output: the atmosphere was very nice and the food is good but i have been to

Input: the staff were very
Output: the staff were very friendly and helpful and the service was great but i would have

Input: the prices were
Output: the prices were very good and the food was great but the service was great

Input: the location is
Output: the location is very nice and the food is good but i have been to

Input: the food was
Output: the food was good but i had a great time here and it was a
